<div style="background:#A6192E;color:#fff;padding:22px 28px;border-radius:14px;font-family:'Barlow Condensed','Arial Narrow',Arial,sans-serif">
<div style="letter-spacing:.2em;text-transform:uppercase;font-size:13px;opacity:.9">Python para Finanzas y Análisis de Datos · Sesión 2</div>
<div style="font-size:34px;font-weight:300;text-transform:uppercase;margin-top:6px"><span style="background:#fff;color:#A6192E;border-radius:10px;padding:2px 12px;margin-right:14px;font-weight:500">2.1</span>Consolidar y auditar datos reales</div>
</div>

**Objetivo.** unir 50 archivos con formatos distintos y detectar errores automáticamente.

**Lo que vamos a hacer**

- Generar 50 archivos de ventas (uno por sucursal) con errores típicos.
- Consolidarlos en una sola tabla.
- Auditar duplicados, valores negativos y datos faltantes.
- Limpiar y resumir por producto y mes.

In [ ]:
import sys, pathlib, subprocess, warnings
warnings.filterwarnings("ignore")

# Localiza la carpeta del curso (en Google Colab clona el repositorio)
root = pathlib.Path.cwd().resolve()
while root != root.parent and not (root / "common").exists():
    root = root.parent
if not (root / "common").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Ulysess824/Python-for-Finance"], check=False)
    root = pathlib.Path("Python-for-Finance/python-finance-course").resolve()
sys.path.insert(0, str(root))
# En Colab, ejecuta antes:  !pip install -q yfinance arch

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_plot_style()

## 1. Los archivos

In [ ]:
from common.demo_data import make_sales_files

folder = root / "data" / "demo_sales"
files = make_sales_files(folder, n_files=50)
print(f"{len(files)} archivos en {folder.name}/")
print(files[0].read_text()[:300])

## 2. Consolidar

> ### ✏️ Ejercicio 1 — Un solo DataFrame
> Lee todos los archivos, pasa los nombres de columnas a minúscula y agrega la columna `origen` (nombre del archivo). Guarda el resultado en `raw`.

In [ ]:
# TODO: raw = ...
raw = None

In [ ]:
frames = []
for f in sorted(files):
    df = pd.read_csv(f)
    df.columns = df.columns.str.lower()
    df["origen"] = f.stem
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)
print(raw.shape)

## 3. Auditar

> ### ✏️ Ejercicio 2 — Reporte de calidad
> Cuenta filas duplicadas, `unidades <= 0` y `precio` faltante. Luego muestra qué archivos tienen problemas.

In [ ]:
# TODO

In [ ]:
audit = pd.Series({
    "filas": len(raw),
    "duplicadas": raw.duplicated().sum(),
    "unidades <= 0": (raw.unidades <= 0).sum(),
    "precio faltante": raw.precio.isna().sum(),
})
display(audit)

flags = raw.assign(dup=raw.duplicated(), neg=raw.unidades <= 0, sin_precio=raw.precio.isna())
by_file = flags.groupby("origen")[["dup", "neg", "sin_precio"]].sum()
display(by_file[by_file.sum(axis=1) > 0])

## 4. Limpiar y resumir

> ### ✏️ Ejercicio 3 — Tabla de ventas por mes
> Elimina los problemas, convierte `fecha` (formatos mixtos, día primero) y crea `ventas = unidades * precio`. Arma `resumen`: ventas por mes (filas) y producto (columnas).

In [ ]:
# TODO: clean = ...   resumen = ...
clean = resumen = None

In [ ]:
clean = (raw.drop_duplicates()
            .query("unidades > 0")
            .dropna(subset=["precio"])
            .assign(fecha=lambda d: pd.to_datetime(d["fecha"], format="mixed", dayfirst=True),
                    ventas=lambda d: d["unidades"] * d["precio"]))
resumen = clean.pivot_table(index=clean["fecha"].dt.to_period("M"), columns="producto",
                            values="ventas", aggfunc="sum")
print(f"{len(raw)} filas → {len(clean)} limpias")
display((resumen / 1e6).round(2))

In [ ]:
(resumen / 1e6).plot.bar(stacked=True, figsize=(9, 4), title="Ventas por mes y producto (millones)")
plt.xticks(rotation=0); plt.legend(loc="upper left", bbox_to_anchor=(1, 1)); plt.show()

## Para llevarse

- Un bucle de 5 líneas reemplaza abrir, copiar y pegar 50 archivos, y se repite igual el próximo mes.
- Auditar con código deja evidencia: sabes qué se corrigió y dónde.
- El mismo script sirve para 500 archivos.

---
*Excel para mirar, Python para repetir.*